# RAG Pipeline Demo

This notebook demonstrates the Retrieval-Augmented Generation (RAG) pipeline step by step using the project's custom modules.

## 1. Import Libraries

Import the custom modules from the src directory that implement the RAG components.

In [1]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import DataLoader
from src.embedding_service import EmbeddingService
from src.vector_store import VectorStore
from src.retriever import Retriever
from src.generator import Generator
from src.pipeline import RAGPipeline

/Users/lukasz/Documents/projects/rags/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Prepare Document Data

Load and prepare the document data using the DataLoader class, which loads sample data from the data directory.

In [2]:
# Load sample data
loader = DataLoader()
documents = loader.load_sample_data()
print(f"Loaded {len(documents)} documents")
print("Sample document:", documents[0] if documents else "No documents")

Loaded 12 documents
Sample document: {'topic': 'Technologia', 'content': 'Sztuczna inteligencja to dziedzina nauki zajmująca się tworzeniem maszyn i systemów, które mogą wykonywać zadania wymagające ludzkiej inteligencji. Ki jest używana w wielu aplikacjach takich jak systemy rekomendacyjne, chatboty i analiza danych.'}


## 3. Build Vector Database Index

Create vector embeddings for the documents using the EmbeddingService and build a FAISS-based vector store for efficient similarity search.

In [3]:
# Create embeddings
embedding_service = EmbeddingService()
documents_list = [d["content"] for d in documents]
metadata_list = [{"topic": d["topic"], "id": i} for i, d in enumerate(documents)]
embeddings = embedding_service.embed(documents_list)

# Build vector store
embedding_dim = embedding_service.get_dimension()
vector_store = VectorStore(embedding_dim)
vector_store.add_documents(embeddings, documents_list, metadata_list)
print(f"Vector store built with {vector_store.get_size()} documents")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6068.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store built with 12 documents


## 4. Implement Retrieval Function

Initialize the Retriever component to perform similarity search on the vector store based on a query.

In [4]:
# Set up retriever
retriever = Retriever(embedding_service, vector_store)

# Example retrieval
query = "Co to jest sztuczna inteligencja?"
results = retriever.retrieve(query, top_k=3)
print(f"Retrieved {len(results)} documents for query: {query}")
for i, result in enumerate(results):
    print(f"Document {i + 1}: {result['content'][:100]}... (score: {result['score']:.4f})")
print("\nNote: Score represents similarity (0-1), where 1.0 means most similar.")

Retrieved 3 documents for query: Co to jest sztuczna inteligencja?
Document 1: Sztuczna inteligencja to dziedzina nauki zajmująca się tworzeniem maszyn i systemów, które mogą wyko... (score: 0.5823)
Document 2: Azja jest największym kontynentem pod względem powierzchni i populacji. Obejmuje wiele krajów o różn... (score: 0.5788)
Document 3: Afryka jest drugim co do wielkości kontynentem. Posiada dużą różnorodność geograficzną - od pustyń, ... (score: 0.5080)

Note: Score represents similarity (0-1), where 1.0 means most similar.


## 5. Set Up Language Model for Generation

Initialize the Generator component, which uses a pre-trained language model (e.g., from Hugging Face) for generating responses based on retrieved documents.

In [5]:
# Set up generator
generator = Generator()
print("Generator initialized with language model.")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 3369.88it/s]
Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Generator initialized with language model.


## 6. Create RAG Pipeline Function

Combine all components into the RAGPipeline class, which orchestrates the retrieval and generation process.

In [6]:
# Create full RAG pipeline
pipeline = RAGPipeline()
pipeline.initialize_with_sample_data()
print("RAG Pipeline initialized and ready for queries.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4675.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1690.91it/s]


RAG Pipeline initialized and ready for queries.


## 7. Run Example Query

Run the complete RAG pipeline with a sample query to see retrieval and generation in action.

In [7]:
# Run example query
query = "Co to jest sztuczna inteligencja?"
result = pipeline.query(query, top_k=3)

print("Query:", query)
print("\nGenerated Answer:")
print(result["answer"])
print(f"\nNumber of sources: {result['num_sources']}")
print("\nTop Sources:")
for i, source in enumerate(result["sources"], 1):
    print(f"Source {i} (Score: {source['score']:.4f}): {source['content'][:150]}...")

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Co to jest sztuczna inteligencja?

Generated Answer:
Sztuczna inteligencja to dziedzina nauki zajmująca się tworzeniem maszyn i systemów, które mogą wykonywać zadania wymagające ludzkiej inteligencji.
Sztuczna inteligencja jest używana w wielu aplikacjach takich jak systemy rekomendacyjne, chatboty i analiza danych.
Answer: Sztuczna inteligencja to dziedzina nauki zajmująca się tworzeniem maszyn i systemów, które mogą wykonywać zadania wymagające ludzkiej inteligencji. Sztuczna inteligencja jest używana w wiel

Number of sources: 3

Top Sources:
Source 1 (Score: 0.5823): Sztuczna inteligencja to dziedzina nauki zajmująca się tworzeniem maszyn i systemów, które mogą wykonywać zadania wymagające ludzkiej inteligencji. Ki...
Source 2 (Score: 0.5788): Azja jest największym kontynentem pod względem powierzchni i populacji. Obejmuje wiele krajów o różnych kulturach, tradycjach i systemach politycznych...
Source 3 (Score: 0.5080): Afryka jest drugim co do wielkości kontynentem. Posiada